In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DATA_PATH = "/content/drive/MyDrive/electronics_cleaned_2_part2/reviews_cleaned2_electronics_part2.parquet"

In [ ]:
!cp $DATA_PATH .

In [ ]:
DATA_PATH = "/content/reviews_cleaned2_electronics_part2.parquet"

## Sampling and embedding

In [ ]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile(DATA_PATH)

print(parquet_file.schema)

print(parquet_file.schema_arrow)

required group field_id=-1 root {
  optional binary field_id=-1 parent_asin (String);
  optional binary field_id=-1 review_id (String);
  optional int64 field_id=-1 sentence_id;
  optional binary field_id=-1 sentence_text (String);
  optional double field_id=-1 rating;
}

parent_asin: large_string
review_id: large_string
sentence_id: int64
sentence_text: large_string
rating: double


In [ ]:
import polars as pl

df = (
    pl.scan_parquet(DATA_PATH)
    .select(["parent_asin", "sentence_id", "sentence_text", "rating"])
    .collect(engine="streaming")
    .sample(n=1_000_000, seed=42, shuffle=True)
)

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
import os

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

model_id = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
def encode_batch(sentences):
    encoded_input = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        model_output = model(**encoded_input)

    sentence_embeddings = mean_pooling(model_output, encoded_input["attention_mask"])
    sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

    return sentence_embeddings.cpu().tolist()

In [ ]:
import gc
from pathlib import Path
import pyarrow.parquet as pq
from tqdm import tqdm


def write_embeddings_from_sampled_df(
    df: pl.DataFrame,
    output_path: str,
    batch_size: int = 4096,
    device="cuda",
    max_length: int = 256,
):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_path.exists():
        output_path.unlink()

    writer = None

    try:
        for start in tqdm(range(0, df.height, batch_size)):
            batch_df = df.slice(start, batch_size)

            sentences = batch_df["sentence_text"].to_list()
            embeddings = encode_batch(sentences)

            out_df = batch_df.with_columns(
                pl.Series("embedding", embeddings, dtype=pl.List(pl.Float32))
            )

            table = out_df.to_arrow()

            if writer is None:
                writer = pq.ParquetWriter(
                    str(output_path),
                    table.schema,
                    compression="zstd",
                )

            writer.write_table(table)

            del batch_df, sentences, embeddings, out_df, table
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    finally:
        if writer is not None:
            writer.close()

In [ ]:
write_embeddings_from_sampled_df(
    df=df,
    output_path="/content/tmp_embeddings.parquet",
    batch_size=1024,
    device=device,
    max_length=256,
)

100%|██████████| 977/977 [28:52<00:00,  1.77s/it]


## Clustering, then sampling

In [ ]:
import numpy as np

def normalize_rows(x):
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(norms, 1e-12)

In [ ]:
import pyarrow as pa

def embedding_array_from_arrow(column):
    if isinstance(column, pa.ChunkedArray):
        column = column.combine_chunks()

    values = column.to_pylist()
    x = np.asarray(values, dtype=np.float32)

    if x.ndim != 2:
        raise ValueError(f"Embedding column must be 2D, got shape={x.shape}")

    return x

In [ ]:
import pyarrow.parquet as pq

def iter_parquet_batches(parquet_path, columns, batch_size=1024):
    pf = pq.ParquetFile(parquet_path)
    yield from pf.iter_batches(batch_size=batch_size, columns=columns)

In [ ]:
from sklearn.cluster import MiniBatchKMeans

def create_model(n_clusters, batch_size: int = 4096, random_state = 42):
    return MiniBatchKMeans(
        n_clusters=n_clusters,
        init="k-means++",
        batch_size=batch_size,
        n_init="auto",
        reassignment_ratio=0.01,
        max_no_improvement=20,
        random_state=random_state,
    )

In [ ]:
from tqdm import tqdm

def train_minibatch_kmeans_on_parquet(
    parquet_path,
    n_clusters,
    embedding_col="embedding",
    batch_size=1024,
    n_epochs=5,
    normalize=True,
    random_state=42,
):
    model = create_model(
        n_clusters=n_clusters,
        batch_size=batch_size,
        random_state=random_state,
    )

    for epoch in range(n_epochs):
        seen = 0

        for batch in tqdm(iter_parquet_batches(
            parquet_path=parquet_path,
            columns=[embedding_col],
            batch_size=batch_size,
        )):
            x = embedding_array_from_arrow(batch.column(0))

            if x.shape[0] == 0:
                continue

            if normalize:
                x = normalize_rows(x)

            model.partial_fit(x)
            seen += x.shape[0]

        print(f"[train] epoch={epoch + 1}/{n_epochs}, processed_rows={seen}")

    return model

In [ ]:
from pathlib import Path

def predict_clusters_to_parquet(
    parquet_path,
    output_path,
    model,
    embedding_col="embedding",
    batch_size=1024,
    normalize=True,
    compression="snappy",
):
    output_path = Path(output_path)
    writer = None
    total_rows = 0

    pf = pq.ParquetFile(parquet_path)
    all_cols = pf.schema_arrow.names

    if embedding_col not in all_cols:
        raise ValueError(f"Column '{embedding_col}' not found in parquet file")

    output_cols = [col for col in all_cols if col != embedding_col]

    for batch in pf.iter_batches(batch_size=batch_size, columns=all_cols):
        table = pa.Table.from_batches([batch])

        x = embedding_array_from_arrow(table[embedding_col])

        if x.shape[0] == 0:
            continue

        if normalize:
            x = normalize_rows(x)

        cluster_ids = model.predict(x)

        out_arrays = []
        out_names = []

        for col in output_cols:
            out_arrays.append(table[col])
            out_names.append(col)

        out_arrays.append(pa.array(cluster_ids, type=pa.int32()))
        out_names.append("cluster_id")

        out_table = pa.table(out_arrays, names=out_names)

        if writer is None:
            writer = pq.ParquetWriter(
                where=str(output_path),
                schema=out_table.schema,
                compression=compression,
            )

        writer.write_table(out_table)
        total_rows += out_table.num_rows

    if writer is not None:
        writer.close()

    print(f"[predict] wrote_rows={total_rows} -> {output_path}")

In [ ]:
parquet_path = "/content/drive/MyDrive/embedding_1m_samples_Electronics_part2/tmp_embeddings.parquet"
output_path = "/content/drive/MyDrive/embedding_1m_samples_Electronics_part2/clustered.parquet"

passthrough_cols = ["doc_id", "title", "source"]

model = train_minibatch_kmeans_on_parquet(
    parquet_path=parquet_path,
    n_clusters=100,
    normalize=True,
    random_state=42,
)

predict_clusters_to_parquet(
    parquet_path=parquet_path,
    output_path=output_path,
    model=model,
    normalize=True,
)

977it [03:56,  4.12it/s]


[train] epoch=1/5, processed_rows=1000000


977it [03:56,  4.14it/s]


[train] epoch=2/5, processed_rows=1000000


977it [03:54,  4.16it/s]


[train] epoch=3/5, processed_rows=1000000


977it [03:57,  4.12it/s]


[train] epoch=4/5, processed_rows=1000000


977it [03:53,  4.18it/s]


[train] epoch=5/5, processed_rows=1000000
[predict] wrote_rows=1000000 -> /content/drive/MyDrive/embedding_1m_samples_Electronics_part2/clustered.parquet


In [ ]:
import polars as pl
import os

input_path = "/content/drive/MyDrive/embedding_1m_samples_Electronics_part2/clustered.parquet"
output_dir = "/content/drive/MyDrive/embedding_1m_samples_Electronics_part2/"

df_sampled = (
    pl.read_parquet(input_path)
    .group_by("cluster_id")
    .map_groups(lambda group: group.sample(n=min(len(group), 8), seed=42))
)

final_df = (
    df_sampled
    .drop("cluster_id")
    .with_columns([
        pl.lit("[]").alias("triplets"),
        pl.lit("electronics_p2").alias("category_name")
    ])
)

chunk_size = 100
for i in range(8):
    start_idx = i * chunk_size
    chunk = final_df.slice(start_idx, chunk_size)

    chunk_path = os.path.join(output_dir, f"samples_chunk_{i}.csv")
    chunk.write_csv(chunk_path)
    print(f"Saved chunk {i} to {chunk_path}")

Saved chunk 0 to /content/drive/MyDrive/embedding_1m_samples_Electronics_part2/samples_chunk_0.csv
Saved chunk 1 to /content/drive/MyDrive/embedding_1m_samples_Electronics_part2/samples_chunk_1.csv
Saved chunk 2 to /content/drive/MyDrive/embedding_1m_samples_Electronics_part2/samples_chunk_2.csv
Saved chunk 3 to /content/drive/MyDrive/embedding_1m_samples_Electronics_part2/samples_chunk_3.csv
Saved chunk 4 to /content/drive/MyDrive/embedding_1m_samples_Electronics_part2/samples_chunk_4.csv
Saved chunk 5 to /content/drive/MyDrive/embedding_1m_samples_Electronics_part2/samples_chunk_5.csv
Saved chunk 6 to /content/drive/MyDrive/embedding_1m_samples_Electronics_part2/samples_chunk_6.csv
Saved chunk 7 to /content/drive/MyDrive/embedding_1m_samples_Electronics_part2/samples_chunk_7.csv


## Agg

In [2]:
import pandas as pd
import os

base_path = "/content/drive/MyDrive/embedding_1m_samples_Electronics_part2/labeled_chunks/"
dfs = []

for i in range(8):
    file_path = os.path.join(base_path, f"samples_chunk_{i}_labeled.csv")
    if os.path.exists(file_path):
        df_chunk = pd.read_csv(file_path)
        dfs.append(df_chunk)
        print(f"Loaded {file_path}")
    else:
        print(f"Warning: {file_path} not found")

final_labeled_df = pd.concat(dfs, ignore_index=True)

print(f"\nTổng số dòng: {len(final_labeled_df)}")
display(final_labeled_df.head())

Loaded /content/drive/MyDrive/embedding_1m_samples_Electronics_part2/labeled_chunks/samples_chunk_0_labeled.csv
Loaded /content/drive/MyDrive/embedding_1m_samples_Electronics_part2/labeled_chunks/samples_chunk_1_labeled.csv
Loaded /content/drive/MyDrive/embedding_1m_samples_Electronics_part2/labeled_chunks/samples_chunk_2_labeled.csv
Loaded /content/drive/MyDrive/embedding_1m_samples_Electronics_part2/labeled_chunks/samples_chunk_3_labeled.csv
Loaded /content/drive/MyDrive/embedding_1m_samples_Electronics_part2/labeled_chunks/samples_chunk_4_labeled.csv
Loaded /content/drive/MyDrive/embedding_1m_samples_Electronics_part2/labeled_chunks/samples_chunk_5_labeled.csv
Loaded /content/drive/MyDrive/embedding_1m_samples_Electronics_part2/labeled_chunks/samples_chunk_6_labeled.csv
Loaded /content/drive/MyDrive/embedding_1m_samples_Electronics_part2/labeled_chunks/samples_chunk_7_labeled.csv

Tổng số dòng: 800


,parent_asin,sentence_id,sentence_text,rating,triplets,category_name
0,B07DGRVTWF,3,basically a poor implementation of the alexa p...,2.0,"[[""alexa platform"", ""poor implementation"", 0]]",electronics_p2
1,B07BFPJ6VX,4,support through direct messaging was great no ...,5.0,"[[""support through direct messaging"", ""great"",...",electronics_p2
2,B0B1PYNH8X,1,for echo show 5 i have the 2nd generation echo...,5.0,[],electronics_p2
3,B0007X9JMA,4,br br note i ve only used it for regular compu...,5.0,[],electronics_p2
4,B00N63V39K,6,wanted to use for business but i m in sales an...,1.0,[],electronics_p2


In [11]:
import ast

def get_triplet_count(val):
    try:
        triplets = ast.literal_eval(val)
        return len(triplets)
    except:
        return 0

final_labeled_df['num_triplets'] = final_labeled_df['triplets'].apply(get_triplet_count)

empty_rows = (final_labeled_df['num_triplets'] == 0).sum()

total_triplets = final_labeled_df['num_triplets'].sum()

print(f"Số hàng không có triplets: {empty_rows}")
print(f"Tổng số lượng triplets tìm được: {total_triplets}")

Số hàng không có triplets: 224
Tổng số lượng triplets tìm được: 862


In [12]:
final_labeled_df.num_triplets.value_counts()

,count
num_triplets,
1,359
0,224
2,163
3,41
4,11
5,2


In [14]:
final_labeled_df.drop(columns=['num_triplets']).to_csv("/content/drive/MyDrive/embedding_1m_samples_Electronics_part2/labeled.csv", encoding='utf-8', index=False)

## Test

In [ ]:
import polars as pl

# Đường dẫn tới file vừa tạo
tmp_file = "/content/drive/MyDrive/embedding_1m_samples_Electronics_part2/clustered.parquet"

# Sử dụng lazy scan để tránh load toàn bộ vào RAM
lazy_df = pl.scan_parquet(tmp_file)

print("--- Schema ---")
print(lazy_df.schema)

# Đếm số lượng dòng (Lazy count)
row_count = lazy_df.select(pl.len()).collect().item()
print(f"\n--- Tổng số dòng: {row_count:,} ---")

# Xem 5 dòng đầu tiên
print("\n--- Head (5 rows) ---")
print(lazy_df.head(5).collect())

--- Schema ---
Schema({'parent_asin': String, 'sentence_id': Int64, 'sentence_text': String, 'rating': Float64, 'cluster_id': Int32})


/tmp/ipykernel_3221/1332191214.py:10: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  print(lazy_df.schema)



--- Tổng số dòng: 1,000,000 ---

--- Head (5 rows) ---
shape: (5, 5)
┌─────────────┬─────────────┬─────────────────────────────────┬────────┬────────────┐
│ parent_asin ┆ sentence_id ┆ sentence_text                   ┆ rating ┆ cluster_id │
│ ---         ┆ ---         ┆ ---                             ┆ ---    ┆ ---        │
│ str         ┆ i64         ┆ str                             ┆ f64    ┆ i32        │
╞═════════════╪═════════════╪═════════════════════════════════╪════════╪════════════╡
│ B09QPKVZFG  ┆ 1           ┆ did nt fit i check to make sur… ┆ 1.0    ┆ 47         │
│ B06XW8T7G7  ┆ 3           ┆ the stand is a little hard to … ┆ 2.0    ┆ 45         │
│ B00O3GTV4I  ┆ 11          ┆ br the extras are a nice bonus  ┆ 4.0    ┆ 43         │
│ B08J9JK4H5  ┆ 5           ┆ br br these jvc headphones cov… ┆ 5.0    ┆ 18         │
│ B06XG9K3ZY  ┆ 2           ┆ same as the original battery f… ┆ 5.0    ┆ 10         │
└─────────────┴─────────────┴─────────────────────────────────┴───────

In [ ]:
!cp /content/tmp_embeddings.parquet /content/drive/MyDrive/embedding_1m_samples_Electronics_part2/